In [1]:
import os
%pwd

'c:\\Users\\YOGA\\OneDrive\\Documents\\AI Projects\\Text-Summarizer\\research'

In [2]:
os.chdir("../")
%pwd

'c:\\Users\\YOGA\\OneDrive\\Documents\\AI Projects\\Text-Summarizer'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainingConfig:
    rootDir:Path
    dataPath:Path
    modelCkpt:Path
    numTrainEpoche: int
    warmupStreps: int
    perDeviceTrainBatchSize: int
    weightDecay: float
    loggingSteps: int
    evaluationStratergy: str
    evalSteps: int
    saveSteps: float
    gradientAccumulationSteps: int

In [4]:
from src.text_summarizer.constants import *
from src.text_summarizer.utils.common import readYaml,createDir

In [5]:
class ConfigManager:
    def __init__(self,
                 config_path=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):
        self.config=readYaml(config_path)
        self.params=readYaml(params_filepath)

        createDir([self.config.artifactsRoots])

    def getModelTrainingConfig(self)-> ModelTrainingConfig:
        config= self.config.modelTraining
        params=self.params.TrainingArguments
        
        createDir([config.rootDir])
        modelTrainingConfig=ModelTrainingConfig(
            rootDir=config.rootDir,
            dataPath=config.dataPath,
            modelCkpt=config.modelCkpt,
            numTrainEpoche=params.numTrainEpoche,
            warmupStreps=params.warmupStreps,
            perDeviceTrainBatchSize=params.perDeviceTrainBatchSize,
            weightDecay=params.weightDecay,
            loggingSteps=params.loggingSteps,
            evaluationStratergy=params.evaluationStratergy,
            evalSteps=params.evalSteps,
            saveSteps=params.saveSteps,
            gradientAccumulationSteps=params.gradientAccumulationSteps
            
        )
        return modelTrainingConfig


In [6]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import  TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
import torch
from datasets import load_from_disk

c:\Users\YOGA\OneDrive\Documents\AI Projects\Text-Summarizer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class ModelTraining:
    def __init__(self, config:ModelTrainingConfig):
        self.config= config
    
    def train(self):
        device="cuda" if torch.cuda.is_available() else "cpu"
        tokenizer=AutoTokenizer.from_pretrained(self.config.modelCkpt)
        modelPegasus=AutoModelForSeq2SeqLM.from_pretrained(self.config.modelCkpt)
        seq2seqDataCollator= DataCollatorForSeq2Seq(tokenizer,model=modelPegasus)

        #load data
        datasetSamsumPt=load_from_disk(self.config.dataPath)

        trainingArgs=TrainingArguments(
            output_dir= self.config.rootDir, num_train_epochs= 1, warmup_steps=500,
            per_device_train_batch_size= 1, per_device_eval_batch_size= 1,
            weight_decay=0.01, logging_steps=10,
            eval_steps=500, save_steps=1e6,
            gradient_accumulation_steps=16
            
        )

        trainer= Trainer(model=modelPegasus, args=trainingArgs,
                         processing_class=tokenizer,data_collator=seq2seqDataCollator,
                         train_dataset=datasetSamsumPt["test"],
                         eval_dataset=datasetSamsumPt["validation"])
        trainer.train()

        #save model
        modelPegasus.save_pretrained(os.path.join(self.config.rootDir,"pegasus-samsum-model"))
        ##Save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.rootDir,"tokenizer"))

In [8]:
config= ConfigManager()
modelTrainingConfig=config.getModelTrainingConfig()
modelTraining=ModelTraining(config=modelTrainingConfig)
modelTraining.train()




[2026-03-06 14:52:27,938: INFO: common: yaml file:config\config.yaml loaded sucessesful]


[2026-03-06 14:52:27,968: INFO: common: yaml file:params.yaml loaded sucessesful]
[2026-03-06 14:52:27,972: INFO: common: created dir at:artifacts]
[2026-03-06 14:52:27,972: INFO: common: created dir at:artifacts/modelTrainer]
[2026-03-06 14:52:28,866: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]


[2026-03-06 14:52:28,866: WARNING: _http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-03-06 14:52:28,877: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-03-06 14:52:29,182: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-06 14:52:29,198: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-03-06 14:52:29,464: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Foun

Loading weights: 100%|██████████| 680/680 [00:02<00:00, 241.19it/s, Materializing param=model.shared.weight]                                   
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:th

[2026-03-06 14:52:47,982: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-06 14:52:48,001: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/generation_config.json "HTTP/1.1 200 OK"]


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
c:\Users\YOGA\OneDrive\Documents\AI Projects\Text-Summarizer\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,189.577051
20,187.950549
30,187.601941
40,184.098193
50,182.475671


Writing model shards: 100%|██████████| 1/1 [00:41<00:00, 41.32s/it]
